
> ## 🏆 최종 결론 — 채택 모델: **정형 Statcast XGBoost (pitch15, 59 features)**
>
> | 항목 | 값 |
> |---|---|
> | Y 지표 | whiff% (헛스윙률) |
> | X 구간 | pitch15 (초반 15구) |
> | 모델 | XGBoost (Optuna 튜닝) |
> | **Test(2025) R²** | **0.0344** |
> | Test RMSE / MAE | 0.0885 / 0.0705 |
>
> - 베이스라인(wOBA·batter9, R²≈0.007) 대비 **약 5배 개선**.
> - 발전 시도(14번 도메인 feature, 15번 시퀀스 CNN)는 모두 검증 통과 못 함 → **본 정형 XGB가 챔피언**.


# 13. 최종 평가 및 해석

**선행 조건**: 선행 단계 완료 → 최종 모델 pkl 파일 존재

**목적**
1. Test(2025) 최종 성능 평가 — Val로 선택한 모델이 실제 미래 데이터에서도 유효한지
2. SHAP 심층 분석 — beeswarm, dependency, waterfall
3. 투수 타입별(구속/구종) 예측 정확도 분석
4. 정형 단독 vs 정형+영상 최종 비교 (있는 경우)
5. 전체 실험 파이프라인 결과 요약

| 단계 | 내용 |
|---|---|
| 1 | Test set 성능 평가 |
| 2 | SHAP 심층 시각화 |
| 3 | 투수 타입별 오차 분석 |
| 4 | 정형 vs 융합 최종 비교 |
| 5 | 전체 파이프라인 요약 |

In [ ]:
import os, sys

IN_COLAB = os.path.exists('/content')

if IN_COLAB:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/투수 컨디션 예측 ML'
else:
    DRIVE = r'c:\Users\suyou\OneDrive\Desktop\ASAC\PROJECT\투수 컨디션 예측'

FEAT_DIR   = os.path.join(DRIVE, '0_data', '4_features')
OUTPUT_DIR = os.path.join(DRIVE, '4_output')
IMAGE_DIR  = os.path.join(DRIVE, '4_output', 'final_images')
os.makedirs(IMAGE_DIR, exist_ok=True)

# 최종 모델 파일 확인
MODEL_CANDIDATES = [
    os.path.join(OUTPUT_DIR, 'fused_final_model.pkl'),     # 정형+영상 (Phase 9)
    os.path.join(OUTPUT_DIR, 'statcast_final_model.pkl'),  # 정형만 (Phase 8)
]
STATCAST_MODEL_PATH = os.path.join(OUTPUT_DIR, 'statcast_final_model.pkl')
FUSED_MODEL_PATH    = os.path.join(OUTPUT_DIR, 'fused_final_model.pkl')

for path in MODEL_CANDIDATES:
    print(f'{os.path.basename(path)}: {"✅" if os.path.exists(path) else "❌"}')

HAS_BIO = os.path.exists(FUSED_MODEL_PATH)
print(f'\n영상 기반 융합 모델 사용: {HAS_BIO}')

In [ ]:
try:
    import xgboost, shap
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'xgboost', 'shap', '-q'])

import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import joblib
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')
print('모듈 로드 완료')

## 1. 데이터 및 최종 모델 로드

In [ ]:
META_COLS = ['game_pk', 'pitcher', 'season', 'y_whiff']

# Statcast feature 로드
df_stat = pd.read_parquet(os.path.join(FEAT_DIR, 'features_pitch15.parquet'))
STAT_COLS = [c for c in df_stat.columns if c not in META_COLS]

# 영상 feature 로드 (있는 경우)
BIO_PATH = os.path.join(FEAT_DIR, 'features_bio_pitch15.parquet')
if HAS_BIO and os.path.exists(BIO_PATH):
    df_bio  = pd.read_parquet(BIO_PATH)
    BIO_COLS = [c for c in df_bio.columns if c not in META_COLS]
    MERGE_KEYS = ['game_pk', 'pitcher', 'season']
    df = df_stat.merge(
        df_bio[[c for c in df_bio.columns if c != 'y_whiff']],
        on=MERGE_KEYS, how='inner'
    )
    FINAL_COLS = STAT_COLS + BIO_COLS
    print(f'융합 데이터 사용: {df.shape}')
else:
    df = df_stat
    BIO_COLS   = []
    FINAL_COLS = STAT_COLS
    print(f'Statcast 단독 데이터 사용: {df.shape}')

# feature cols 파일에서 읽기 (Phase 8~9에서 저장)
feat_csv = os.path.join(OUTPUT_DIR, 'fused_feature_cols.csv' if HAS_BIO else 'final_feature_cols.csv')
if os.path.exists(feat_csv):
    FINAL_COLS = pd.read_csv(feat_csv)['0'].tolist()
    print(f'feature list 로드: {len(FINAL_COLS)}개')

# train/val/test 분리
train = df[df['season'].isin([2021, 2022, 2023])]
val   = df[df['season'] == 2024]
test  = df[df['season'] == 2025]

X_train, y_train = train[FINAL_COLS], train['y_whiff']
X_val,   y_val   = val[FINAL_COLS],   val['y_whiff']
X_test,  y_test  = test[FINAL_COLS],  test['y_whiff']

# 최종 모델 로드
model_path = FUSED_MODEL_PATH if HAS_BIO else STATCAST_MODEL_PATH
final_model = joblib.load(model_path)
print(f'\n모델 로드: {os.path.basename(model_path)}')
print(f'Train: {len(train):,}  Val: {len(val):,}  Test: {len(test):,}  Features: {len(FINAL_COLS)}')

## 2. Test Set 최종 성능 평가

In [ ]:
splits = {
    'Train (2021-23)': (X_train, y_train),
    'Val   (2024)':    (X_val, y_val),
    'Test  (2025)':    (X_test, y_test),
}

eval_rows = []
for name, (X, y) in splits.items():
    pred = final_model.predict(X)
    row  = {
        'split': name,
        'RMSE':  round(mean_squared_error(y, pred) ** 0.5, 4),
        'MAE':   round(mean_absolute_error(y, pred), 4),
        'R²':    round(r2_score(y, pred), 4),
        'n':     len(y),
    }
    eval_rows.append(row)

eval_df = pd.DataFrame(eval_rows)
print('=== 최종 모델 성능 ===')
print(eval_df.to_string(index=False))

# 시각화: 예측 vs 실제
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, (X, y), color) in zip(axes, [
    ('Val (2024)',  (X_val,  y_val),  'steelblue'),
    ('Test (2025)', (X_test, y_test), 'darkorange'),
]):
    pred = final_model.predict(X)
    ax.scatter(y, pred, alpha=0.4, s=20, color=color)
    lim = [min(y.min(), pred.min()), max(y.max(), pred.max())]
    ax.plot(lim, lim, 'k--', lw=1)
    ax.set_xlabel('실제 whiff%')
    ax.set_ylabel('예측 whiff%')
    ax.set_title(f'{name}  R²={r2_score(y, pred):.4f}  RMSE={mean_squared_error(y, pred)**0.5:.4f}')

plt.suptitle('최종 모델 예측 vs 실제', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(IMAGE_DIR, '01_pred_vs_actual.png'), dpi=150, bbox_inches='tight')
plt.show()

## 3. SHAP 심층 분석

In [ ]:
explainer   = shap.TreeExplainer(final_model)
shap_values = explainer(X_test)
shap_mean   = pd.Series(np.abs(shap_values.values).mean(axis=0), index=FINAL_COLS).sort_values(ascending=False)

print('Top 15 feature SHAP importance:')
print(shap_mean.head(15).round(5).to_string())

In [ ]:
# Beeswarm plot
plt.figure(figsize=(10, 8))
shap.plots.beeswarm(shap_values, max_display=20, show=False)
plt.title('SHAP Beeswarm — Test(2025)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(IMAGE_DIR, '02_shap_beeswarm.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 1위 feature dependency plot
top1 = shap_mean.index[0]
top2 = shap_mean.index[1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
shap.dependence_plot(top1, shap_values.values, X_test, interaction_index=top2, ax=axes[0], show=False)
axes[0].set_title(f'SHAP Dependence: {top1}')
shap.dependence_plot(top2, shap_values.values, X_test, interaction_index=top1, ax=axes[1], show=False)
axes[1].set_title(f'SHAP Dependence: {top2}')

plt.tight_layout()
plt.savefig(os.path.join(IMAGE_DIR, '03_shap_dependence.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# feature 유형별 SHAP 기여도
def classify_feat(name):
    if 'delta' in name: return 'delta (변화량)'
    if name.startswith('avg_'):  return 'avg (절대값)'
    if name.startswith('prev_'): return 'prev (기준값)'
    if name.startswith('std_'):  return 'std (변동성)'
    if any(n in name for n in BIO_COLS): return 'bio (생체역학)'
    return 'other'

type_shap = pd.DataFrame({'feature': FINAL_COLS, 'shap': shap_mean.values})
type_shap['type'] = type_shap['feature'].apply(classify_feat)
type_summary = type_shap.groupby('type')['shap'].agg(['mean', 'sum', 'count']).round(5)
print('feature 유형별 SHAP 기여도:')
print(type_summary.sort_values('mean', ascending=False))

plt.figure(figsize=(8, 4))
type_summary['mean'].sort_values().plot(kind='barh', color='steelblue')
plt.xlabel('평균 |SHAP|')
plt.title('feature 유형별 평균 SHAP 기여도')
plt.tight_layout()
plt.savefig(os.path.join(IMAGE_DIR, '04_shap_by_type.png'), dpi=150, bbox_inches='tight')
plt.show()

# SHAP 결과 저장
shap_df = pd.DataFrame({'feature': FINAL_COLS, 'mean_shap': shap_mean.values,
                        'rank': range(1, len(FINAL_COLS)+1), 'type': type_shap['type'].values})
shap_df.to_csv(os.path.join(OUTPUT_DIR, 'final_shap_importance.csv'), index=False)

## 4. 투수 타입별 예측 오차 분석

In [ ]:
test_df = test.copy()
test_df['pred'] = final_model.predict(X_test)
test_df['abs_err'] = (test_df['pred'] - test_df['y_whiff']).abs()
test_df['sq_err']  = (test_df['pred'] - test_df['y_whiff']) ** 2

# 구속 기반 타입 분류 (avg_speed_Fastball 이용)
speed_col = 'avg_speed_Fastball'
if speed_col in test_df.columns:
    test_df['speed_type'] = pd.cut(
        test_df[speed_col],
        bins=[0, 90, 93, 96, 120],
        labels=['느림(<90)', '보통(90-93)', '빠름(93-96)', '파이어볼러(96+)']
    )
    speed_group = test_df.groupby('speed_type', observed=True).agg(
        n=('y_whiff', 'count'),
        mean_whiff=('y_whiff', 'mean'),
        rmse=('sq_err', lambda x: x.mean()**0.5),
        mae=('abs_err', 'mean')
    ).round(4)
    print('=== 구속 타입별 오차 분석 ===')
    print(speed_group.to_string())

    plt.figure(figsize=(10, 4))
    speed_group['rmse'].plot(kind='bar', color='steelblue', alpha=0.8)
    plt.xlabel('구속 타입')
    plt.ylabel('RMSE')
    plt.title('구속 타입별 예측 RMSE')
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig(os.path.join(IMAGE_DIR, '05_error_by_speed_type.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f'{speed_col} 컬럼 없음 — 구속 타입 분석 건너뜀')

# 투수별 평균 오차 (상위/하위 10명)
pitcher_err = test_df.groupby('pitcher')['abs_err'].mean().sort_values(ascending=False)
print(f'\n예측 어려운 투수 Top 10 (MAE 높은 순):')
print(pitcher_err.head(10).round(4))
print(f'\n예측 쉬운 투수 Top 10 (MAE 낮은 순):')
print(pitcher_err.tail(10).round(4))

## 5. 정형 단독 vs 정형+영상 최종 비교

In [ ]:
compare_rows = []

# 정형 모델 (항상 존재)
if os.path.exists(STATCAST_MODEL_PATH):
    m_stat = joblib.load(STATCAST_MODEL_PATH)
    stat_feat_csv = os.path.join(OUTPUT_DIR, 'final_feature_cols.csv')
    stat_cols = pd.read_csv(stat_feat_csv)['0'].tolist() if os.path.exists(stat_feat_csv) else STAT_COLS
    stat_cols = [c for c in stat_cols if c in test.columns]

    p_stat = m_stat.predict(test[stat_cols])
    compare_rows.append({
        '모델': '정형 Statcast 단독',
        'feature 수': len(stat_cols),
        'Test RMSE': round(mean_squared_error(y_test, p_stat)**0.5, 4),
        'Test MAE':  round(mean_absolute_error(y_test, p_stat), 4),
        'Test R²':   round(r2_score(y_test, p_stat), 4),
    })

# 융합 모델
if HAS_BIO:
    p_fused = final_model.predict(X_test)
    compare_rows.append({
        '모델': '정형 + 영상 생체역학',
        'feature 수': len(FINAL_COLS),
        'Test RMSE': round(mean_squared_error(y_test, p_fused)**0.5, 4),
        'Test MAE':  round(mean_absolute_error(y_test, p_fused), 4),
        'Test R²':   round(r2_score(y_test, p_fused), 4),
    })

if compare_rows:
    cmp_df = pd.DataFrame(compare_rows)
    print('=== 최종 모델 비교 (Test 2025) ===')
    print(cmp_df.to_string(index=False))
    cmp_df.to_csv(os.path.join(OUTPUT_DIR, 'final_model_comparison.csv'), index=False)
else:
    print('비교할 모델 없음 — statcast_final_model.pkl 확인 필요')

## 6. 전체 파이프라인 실험 결과 요약

In [ ]:
# 각 phase 결과 CSV 통합
summaries = {}
result_files = {
    'x_interval':  os.path.join(OUTPUT_DIR, 'x_interval_experiment_results.csv'),
    'outlier':     os.path.join(OUTPUT_DIR, 'outlier_experiment_results.csv'),
    'tuning':      os.path.join(OUTPUT_DIR, 'tuning_results.csv'),
    'feat_sel':    os.path.join(OUTPUT_DIR, 'feature_selection_results.csv'),
    'bio':         os.path.join(OUTPUT_DIR, 'bio_experiment_results.csv'),
    'final_cmp':   os.path.join(OUTPUT_DIR, 'final_model_comparison.csv'),
}

print('=== 저장된 실험 결과 파일 현황 ===')
for name, path in result_files.items():
    exists = os.path.exists(path)
    print(f'  {"✅" if exists else "❌"}  {os.path.basename(path)}')

# 최종 성능 요약 출력
y_pred_final = final_model.predict(X_test)
final_rmse = mean_squared_error(y_test, y_pred_final) ** 0.5
final_r2   = r2_score(y_test, y_pred_final)

print(f'''
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  최종 모델 성능 (Test 2025)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Y 지표   : whiff% (헛스윙률)
  X 구간   : pitch15 (초반 15구)
  모델     : XGBoost (Optuna 튜닝)
  Test RMSE: {final_rmse:.4f}
  Test R²  : {final_r2:.4f}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  베이스라인 R² (wOBA, batter9) : ~0.007
  최종 R²   (whiff%, pitch15)  : {final_r2:.4f}
  개선율 : {final_r2 / 0.007:.1f}배↑
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
''')

print('이미지 저장 위치:', IMAGE_DIR)
print('완료된 이미지:')
for f in sorted(os.listdir(IMAGE_DIR)):
    print(f'  {f}')

## 7. E9 오차 심층 분석 — "언제 잘 맞고 언제 못 맞나" (2026-07-12)

정형 전체(~23,225경기)로 채택 모델 진단. R²가 낮은 게 (a)전 구간 노이즈인가 (b)특정 구간만 망치나 판별.
4각도: ① 실제 whiff% 구간별 ② 투수 유형별 ③ 상황별 ④ 오차 큰 경기 공통점.

> **주의**: ①의 whiff 구간별 R²는 타겟값으로 나눈 그룹이라 R²가 음수로 폭발함(통계 착시, 신뢰 X). 실제 신호는 **②투수 유형별**(파워형 R²+0.018 vs 핀포인트형 -0.024 → "파워형에만 이 모델 써라")과 **bias 일관 -0.013(과대예측 → 예측값-0.013 보정 가능)**.

In [ ]:
# E9. 정형 모델 오차 분석 — "언제 잘 맞고 언제 못 맞나" (자립형, 정형 전체 사용)
E9_BEST = {'n_estimators':300,'learning_rate':0.05,'max_depth':6,'subsample':0.8,
           'colsample_bytree':0.8,'min_child_weight':1,'reg_alpha':0.0,'reg_lambda':1.0,
           'random_state':42,'n_jobs':-1,'verbosity':0,'early_stopping_rounds':50}

e9_df = pd.read_parquet(os.path.join(FEAT_DIR, 'features_pitch15.parquet'))
E9_META = ['game_pk','pitcher','season','y_whiff']
E9_COLS = [c for c in e9_df.columns if c not in E9_META]
e9_tr = e9_df[e9_df['season'].isin([2021,2022,2023])]
e9_vl = e9_df[e9_df['season']==2024]
e9_te = e9_df[e9_df['season']==2025]

e9_m = xgb.XGBRegressor(**E9_BEST)
e9_m.fit(e9_tr[E9_COLS], e9_tr['y_whiff'], eval_set=[(e9_vl[E9_COLS], e9_vl['y_whiff'])], verbose=False)

ev = e9_te.copy()
ev['_pred'] = e9_m.predict(ev[E9_COLS])
ev['_err']  = ev['y_whiff'] - ev['_pred']
ev['_abs']  = ev['_err'].abs()
print(f'정형 전체 Test R²={r2_score(ev["y_whiff"],ev["_pred"]):.4f}  '
      f'RMSE={mean_squared_error(ev["y_whiff"],ev["_pred"])**0.5:.4f}  n={len(ev)}')

def _gm(mask, name, min_n=40):
    g = ev[mask]
    if len(g) < min_n: return None
    return {'group':name,'n':len(g),'R2':round(r2_score(g['y_whiff'],g['_pred']),4),
            'RMSE':round(mean_squared_error(g['y_whiff'],g['_pred'])**0.5,4),
            'bias(실제-예측)':round(g['_err'].mean(),4)}

rows=[]
# [1] 실제 whiff% 구간별 (R² 착시 주의 — bias만 참고)
qs = ev['y_whiff'].quantile([0.2,0.4,0.6,0.8]).values
ev['_wbin']=pd.cut(ev['y_whiff'], bins=[-np.inf,*qs,np.inf],
                   labels=['최하위20%','하위20-40%','중간','상위60-80%','최상위20%'])
for lb in ev['_wbin'].cat.categories:
    rows.append(_gm(ev['_wbin']==lb, f'whiff {lb}'))
# [2] 투수 유형별 (★ 핵심 신호)
for feat,hn,ln in [('avg_speed_Fastball','파워형(FB구속>med)','핀포인트형(FB구속<=med)'),
                   ('std_speed_all','구속변동큰','구속안정'),
                   ('total_pitches','많이던진','적게던진')]:
    if feat in ev.columns:
        md=e9_tr[feat].median()
        rows.append(_gm(ev[feat]>md, hn)); rows.append(_gm(ev[feat]<=md, ln))
# [3] 상황별
for c in ['p_throws','hand','stand']:
    if c in ev.columns:
        for v in ev[c].dropna().unique(): rows.append(_gm(ev[c]==v, f'{c}={v}'))

res=pd.DataFrame([r for r in rows if r])
print('\n=== Subgroup별 R²/RMSE/bias (test, n>=40) ===')
print(res.sort_values('R2',ascending=False).to_string(index=False))

# [4] 오차 큰 경기 공통점
q80=ev['_abs'].quantile(0.8); big=ev[ev['_abs']>q80]; small=ev[ev['_abs']<=q80]
cc=[c for c in ['y_whiff','total_pitches','avg_speed_Fastball','std_speed_all','strike_ratio'] if c in ev.columns]
cmp=pd.DataFrame({'오차큰경기':big[cc].mean(),'오차작은경기':small[cc].mean()}).round(3)
cmp['차이']=(cmp['오차큰경기']-cmp['오차작은경기']).round(3)
print(f'\n[4] 오차 상위20%(n={len(big)}) vs 나머지 피처 비교:'); print(cmp.to_string())

res.to_csv(os.path.join(OUTPUT_DIR,'error_analysis_subgroups.csv'), index=False)
print('\n저장: error_analysis_subgroups.csv')
print('[결론] 파워형 R²>핀포인트형(음수) → 유형으로 적용범위 좁히면 실용 / bias -0.013 보정가능 / whiff구간R²는 착시라 무시')

## 8. E11 — ACWR(최근 투구량) feature 최종 반영 (2026-07-15)

"컨디션=개인 기준선 대비 델타" 재정의를 5갈래로 검증했으나 전부 기각(2_experiments/12번 인접 탐색, CLAUDE.md E11 참고).
탐색 부산물로 **최근 투구량(급성:만성 부하, ACWR)** feature가 기존 59개에 없던 신호로 확인되어 `feature_aggregator.py`에 정식 편입.

- `acute_load`(최근 7일 투구량) / `chronic_load`(최근 28일 주당평균) / `acwr`(비율) 3개 추가 → **62 feature**
- 진단 결과 실제 기여는 `acwr`(비율)이 아니라 `acute_load`/`chronic_load`(절대치) — 최근 많이 던진 투수일수록 whiff% 소폭 높음(선수 신뢰도/로테이션 신호에 가까움, "피로" 해석은 주의)
- 100-seed paired t-test(3회 반복 검증): **Val 기준 일관되게 유의 개선(+0.0014~0.0025, p<0.001)**, Test는 실행마다 유의~무의 오갔음(표본 필터링 민감) → Val 기준으로 채택

In [ ]:
# E11. ACWR 포함 62-feature 최종모델 vs 기존 59-feature 모델 비교 + 최종모델 갱신
E11_BEST = E9_BEST  # 동일 하이퍼파라미터로 공정 비교

# 주의: feature_aggregator.build_features()의 기본 산출물(features_pitch<n>.parquet)은
#       A(추세)·D(changepoint) feature까지 포함 — 이들은 이미 기각된 실험군이라 최종모델엔 미포함.
#       따라서 "기존 59개 확정 feature + acute_load/chronic_load/acwr 3개"만 담은
#       features_pitch15_with_acwr.parquet(정식 build_features 결과에서 A/D 제외분 재구성)를 사용.
acwr_path = os.path.join(FEAT_DIR, 'features_pitch15_with_acwr.parquet')
if not os.path.exists(acwr_path):
    print(f'[skip] {acwr_path} 없음 — feature_aggregator로 ACWR 재생성 필요')
else:
    e11_df = pd.read_parquet(acwr_path)
    e11_df = e11_df.dropna(subset=['acwr']).copy()  # ACWR 계산 가능한 표본만(직전 이력 부족한 초기 등판 제외)

    E11_META = ['game_pk', 'pitcher', 'season', 'y_whiff']
    E11_COLS = [c for c in e11_df.columns if c not in E11_META]                       # 62개
    E11_BASE_COLS = [c for c in E11_COLS if c not in ('acute_load', 'chronic_load', 'acwr')]  # 59개

    e11_tr = e11_df[e11_df['season'].isin([2021, 2022, 2023])]
    e11_vl = e11_df[e11_df['season'] == 2024]
    e11_te = e11_df[e11_df['season'] == 2025]

    m_base = xgb.XGBRegressor(**E11_BEST)
    m_base.fit(e11_tr[E11_BASE_COLS], e11_tr['y_whiff'], eval_set=[(e11_vl[E11_BASE_COLS], e11_vl['y_whiff'])], verbose=False)

    m_acwr = xgb.XGBRegressor(**E11_BEST)
    m_acwr.fit(e11_tr[E11_COLS], e11_tr['y_whiff'], eval_set=[(e11_vl[E11_COLS], e11_vl['y_whiff'])], verbose=False)

    for name, m, cols in [('기존 59 feature', m_base, E11_BASE_COLS), ('+ACWR 62 feature', m_acwr, E11_COLS)]:
        r2_v = r2_score(e11_vl['y_whiff'], m.predict(e11_vl[cols]))
        r2_t = r2_score(e11_te['y_whiff'], m.predict(e11_te[cols]))
        print(f'{name:<20} Val R²={r2_v:.4f}  Test R²={r2_t:.4f}')

    print('\n※ 위는 단일 seed 결과 — 채택 근거는 100-seed paired t-test(Val 기준 3회 반복 p<0.001 일관 유의, Test는 실행마다 편차)')

    # 최종모델 갱신: ACWR 포함 버전을 새 최종모델로 저장
    joblib.dump(m_acwr, os.path.join(OUTPUT_DIR, 'statcast_final_model_v2_acwr.pkl'))
    pd.Series(E11_COLS).to_csv(os.path.join(OUTPUT_DIR, 'final_feature_cols_v2_acwr.csv'), index=False)
    print(f'\n[saved] statcast_final_model_v2_acwr.pkl ({len(E11_COLS)} features)')